# 예제 05. batch size와 shuffle 실험
빅데이터프로그래밍 · 5주차

batch size와 shuffle을 바꾸면 출력과 학습이 어떻게 달라지는지 직접 확인합니다.


In [ ]:
import torch
from torch.utils.data import TensorDataset, DataLoader

data = TensorDataset(torch.arange(12).float().reshape(-1, 1))
print("전체 개수:", len(data))


## 1. batch size 바꿔 보기


In [ ]:
for bs in [1, 3, 5, 12, 20]:
    loader = DataLoader(data, batch_size=bs, shuffle=False)
    sizes = [b.shape[0] for (b,) in loader]
    print(f"batch_size {bs:3d}  batch 수 {len(loader):2d}  각 크기 {sizes}")


## 2. shuffle 바꿔 보기


In [ ]:
print("shuffle=False — 매번 같은 순서")
for run in range(2):
    order = []
    for (b,) in DataLoader(data, batch_size=4, shuffle=False):
        order += b.flatten().int().tolist()
    print(" ", order)

print("\nshuffle=True — 매번 다른 순서")
for run in range(2):
    order = []
    for (b,) in DataLoader(data, batch_size=4, shuffle=True):
        order += b.flatten().int().tolist()
    print(" ", order)


## 3. drop_last


In [ ]:
for drop in [False, True]:
    loader = DataLoader(data, batch_size=5, drop_last=drop)
    sizes = [b.shape[0] for (b,) in loader]
    print(f"drop_last={drop}  → {sizes}  (버려진 개수 {12 - sum(sizes)})")


## 4. 학습에 주는 영향
같은 데이터, 같은 epoch 수에서 batch size만 바꿔 손실을 비교합니다.


In [ ]:
import torch.nn as nn

def train(batch_size, epochs=20, lr=0.05, seed=0):
    torch.manual_seed(seed)
    X = torch.randn(200, 2)
    y = X @ torch.tensor([[2.0], [-1.0]]) + 0.5

    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True)
    model = nn.Linear(2, 1)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    losses = []
    for _ in range(epochs):
        total = 0.0
        for bx, by in loader:
            loss = loss_fn(model(bx), by)
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * bx.shape[0]
        losses.append(total / len(X))
    return losses


for bs in [8, 32, 200]:
    losses = train(bs)
    print(f"batch_size {bs:4d}  갱신 횟수/epoch {200 // bs if bs <= 200 else 1:3d}"
          f"  마지막 손실 {losses[-1]:.4f}")


In [ ]:
import matplotlib.pyplot as plt

for bs in [8, 32, 200]:
    plt.plot(train(bs), label=f"batch={bs}")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend()
plt.title("batch size vs loss")
plt.show()


## 5. shuffle을 끄면
데이터가 정렬돼 있을 때 문제가 드러납니다. 정답 순으로 정렬된 데이터를 만들어 봅니다.


In [ ]:
X = torch.randn(200, 2)
y = X @ torch.tensor([[2.0], [-1.0]])
idx = y.flatten().argsort()             # 정답 순으로 정렬
X, y = X[idx], y[idx]

def train_sorted(shuffle, epochs=20):
    torch.manual_seed(0)
    loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=shuffle)
    model = nn.Linear(2, 1)
    opt = torch.optim.SGD(model.parameters(), lr=0.05)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        for bx, by in loader:
            loss = loss_fn(model(bx), by)
            opt.zero_grad(); loss.backward(); opt.step()
    with torch.no_grad():
        return loss_fn(model(X), y).item()

print(f"shuffle=False  최종 손실 {train_sorted(False):.4f}")
print(f"shuffle=True   최종 손실 {train_sorted(True):.4f}")


## 직접 해보기
1. batch size 2로 학습하면 시간이 얼마나 더 걸리나요? (`time` 모듈)
2. 검증 loader에 `shuffle=True` 를 주면 결과가 달라지나요? 왜 그런가요?


In [ ]:
# 여기에 작성하세요
